In [ ]:
#라이브러리와 모델 불러오기
import json
import joblib
import pandas as pd

model = joblib.load("risk_model.pkl")

In [ ]:
# 점수표 => priority의 점수표 dict와 calculate_priority_score() 함수를 그대로 먼저 넣은 후 실행
severity_score_map = {
    "LOW": 25,
    "MEDIUM": 50,
    "HIGH": 75,
    "CRITICAL": 100,
}

attack_vector_score_map = {
    "PHYSICAL": 20,
    "LOCAL": 40,
    "ADJACENT_NETWORK": 70,
    "NETWORK": 100,
}

attack_complexity_score_map = {
    "HIGH": 40,
    "LOW": 100,
}

privileges_score_map = {
    "HIGH": 30,
    "LOW": 70,
    "NONE": 100,
}

user_interaction_score_map = {
    "REQUIRED": 40,
    "NONE": 100,
}


def calculate_priority_score(row):

    cvss_score = float(row["cvss_score"]) * 10

    severity_score = severity_score_map.get(
        row["predicted_severity"], 0
    )

    vector_score = attack_vector_score_map.get(
        row["attack_vector"], 0
    )

    complexity_score = attack_complexity_score_map.get(
        row["attack_complexity"], 0
    )

    privilege_score = privileges_score_map.get(
        row["privileges_required"], 0
    )

    interaction_score = user_interaction_score_map.get(
        row["user_interaction"], 0
    )

    score = (
        cvss_score * 0.50
        + severity_score * 0.20
        + vector_score * 0.10
        + complexity_score * 0.08
        + privilege_score * 0.07
        + interaction_score * 0.05
    )

    return round(score, 2)

In [ ]:
#예측 함수
def predict_risk(
    cve_id: str,
    cvss_score: float,
    attack_vector: str,
    attack_complexity: str,
    privileges_required: str,
    user_interaction: str,
    cwe: str,
    description: str,
) -> dict:

    input_data = pd.DataFrame(
        [
            {
                "attack_vector": attack_vector.upper(),
                "attack_complexity": attack_complexity.upper(),
                "privileges_required": privileges_required.upper(),
                "user_interaction": user_interaction.upper(),
                "cwe": cwe.upper(),
                "description": description,
            }
        ]
    )

    predicted_severity = model.predict(
        input_data
    )[0]

    probabilities = model.predict_proba(
        input_data
    )[0]

    confidence = float(
        probabilities.max()
    )

    result_row = {
        "cvss_score": float(cvss_score),
        "predicted_severity": predicted_severity,
        "attack_vector": attack_vector.upper(),
        "attack_complexity": attack_complexity.upper(),
        "privileges_required": privileges_required.upper(),
        "user_interaction": user_interaction.upper(),
    }

    priority_score = calculate_priority_score(
        result_row
    )

    return {
        "cve_id": cve_id,
        "cvss_score": float(cvss_score),
        "predicted_risk": predicted_severity,
        "confidence": round(confidence, 4),
        "priority_score": priority_score,
    }

In [ ]:
#테스트
result = predict_risk(
    cve_id="CVE-TEST-0001",
    cvss_score=9.8,
    attack_vector="NETWORK",
    attack_complexity="LOW",
    privileges_required="NONE",
    user_interaction="NONE",
    cwe="CWE-89",
    description=(
        "Unauthenticated remote SQL injection "
        "allows execution of arbitrary database queries."
    ),
)

print(result)

{'cve_id': 'CVE-TEST-0001', 'cvss_score': 9.8, 'predicted_risk': 'CRITICAL', 'confidence': 0.7853, 'priority_score': 99.0}


In [ ]:
#json 저장
with open(
    "analysis_result.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        result,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("analysis_result.json 저장 완료")

analysis_result.json 저장 완료
